In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
from tqdm import tqdm
import re
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, precision_score, recall_score 
import lightgbm as lgb
import pickle

In [2]:
df = pd.read_csv('../train_csv/train.csv',index_col=0)
rank_map = {1:6,2:5,3:4,4:3,5:2,6:1}
rank_mod_map = {1:6,2:5,3:4,4:0,5:0,6:0}
df_y = df['着'].map(rank_mod_map).astype(int)
df_x = df.drop(['選手名','着'],axis=1)
x_train, x_vali, y_train, y_vali = train_test_split(df_x, df_y, test_size=0.2, shuffle=False)

train_group = x_train['RaceID'].value_counts(sort = False)
val_group = x_vali['RaceID'].value_counts(sort = False)

x_train = x_train.drop('RaceID', axis=1)
x_vali = x_vali.drop('RaceID', axis=1)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 713323 entries, 0 to 713322
Data columns (total 18 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   艇番       713323 non-null  int64  
 1   選手登番     713323 non-null  int64  
 2   選手名      713323 non-null  object 
 3   年齢       713323 non-null  int64  
 4   支部       713323 non-null  int64  
 5   体重       713323 non-null  int64  
 6   級別       713323 non-null  int64  
 7   全国勝率     713323 non-null  float64
 8   全国2連率    713323 non-null  float64
 9   当地勝率     713323 non-null  float64
 10  当地2連率    713323 non-null  float64
 11  モーター2連率  713323 non-null  float64
 12  ボート2連率   713323 non-null  float64
 13  RaceID   713323 non-null  object 
 14  場所       713323 non-null  int64  
 15  R        713323 non-null  int64  
 16  着        713323 non-null  int64  
 17  展示       713323 non-null  float64
dtypes: float64(7), int64(9), object(2)
memory usage: 119.5+ MB


In [5]:

lgbm_params =  {
    'task': 'train',
    'boosting_type': 'gbdt',
    'objective': 'lambdarank', #←ここでランキング学習と指定！
    'metric': 'ndcg',   # for lambdarank
    'ndcg_eval_at': [1,2,3],  # 3連単を予測したい
    'max_position': 6,  # 競艇は6位までしかない
    'learning_rate': 0.01, 
    'group_column': 13,
    'min_data': 1,
    'min_data_in_bin': 1,
    'random_state': 777,
    #'num_leaves': 31,
   #'max_depth':35,
}

lgtrain = lgb.Dataset(x_train, y_train,  group=train_group)
lgvalid = lgb.Dataset(x_vali, y_vali,group=val_group)

lgb_clf = lgb.train(
    lgbm_params,
    lgtrain,
    num_boost_round=10000,
    valid_sets=[lgtrain, lgvalid],
    valid_names=['train','valid'],
    early_stopping_rounds=1000,
    verbose_eval=100
)

/Users/tojo/Documents/boat/.boat_env/lib/python3.10/site-packages/lightgbm/engine.py:181: UserWarning: 'early_stopping_rounds' argument is deprecated and will be removed in a future release of LightGBM. Pass 'early_stopping()' callback via 'callbacks' argument instead.
  _log_warning("'early_stopping_rounds' argument is deprecated and will be removed in a future release of LightGBM. "
/Users/tojo/Documents/boat/.boat_env/lib/python3.10/site-packages/lightgbm/engine.py:239: UserWarning: 'verbose_eval' argument is deprecated and will be removed in a future release of LightGBM. Pass 'log_evaluation()' callback via 'callbacks' argument instead.
  _log_warning("'verbose_eval' argument is deprecated and will be removed in a future release of LightGBM. "


[LightGBM] [Warning] Unknown parameter: max_position
[LightGBM] [Warning] Unknown parameter: max_position
[LightGBM] [Warning] Auto-choosing row-wise multi-threading, the overhead of testing was 0.029605 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2082
[LightGBM] [Info] Number of data points in the train set: 570658, number of used features: 15
[LightGBM] [Warning] Unknown parameter: max_position
Training until validation scores don't improve for 1000 rounds
[100]	train's ndcg@1: 0.682601	train's ndcg@2: 0.70482	train's ndcg@3: 0.742696	valid's ndcg@1: 0.682588	valid's ndcg@2: 0.70463	valid's ndcg@3: 0.741186
[200]	train's ndcg@1: 0.685394	train's ndcg@2: 0.709022	train's ndcg@3: 0.746183	valid's ndcg@1: 0.685096	valid's ndcg@2: 0.707391	valid's ndcg@3: 0.743956
[300]	train's ndcg@1: 0.688017	train's ndcg@2: 0.711596	train's ndcg@3: 0.748466	valid's ndcg@1: 0.686641	valid

In [6]:
with open('../model/lgb_clf.pickle', 'wb') as f:
    pickle.dump(lgb_clf, f)

In [7]:
y_pred_model = lgb_clf.predict(x_vali ,num_iteration=lgb_clf.best_iteration)

In [8]:
y_pred_model[:12]


array([-0.82446567,  1.17708359, -0.05446667, -2.11287577,  1.76761357,
       -0.74656994,  1.02371653, -0.79731888, -1.67865849, -2.85465225,
        0.41050411,  0.14636665])

In [11]:
#Validデータ的中率の算出
j = 0
solo_count = 0
doub_count = 0
tri_count = 0
trio_count = 0
for i in val_group:
    result = y_pred_model[j:j+i] #グループでの順位
    ans = y_vali.iloc[j:j+i].reset_index() #答え
    result1st = np.argmax(result) #予測の1位

    if len(np.where(result==sorted(result)[::-1][0])[0])>1:
        result2nd = np.where(result==sorted(result)[::-1][1])[0][0]
        result3rd = np.where(result==sorted(result)[::-1][1])[0][1]
    else:
        if i > 1:
            result2nd = np.where(result==sorted(result)[::-1][1])[0][0]
        if i > 2:
            result3rd = np.where(result==sorted(result)[::-1][2])[0][0]
    
    #print(int(ans[ans["着"] == ans.max()[1]].index.values))
    ans1st = int(ans[ans["着"]==ans.max()[1]].index.values)
    and_sort = ans.sort_values("着",ascending=False)['着']
    if i > 2:
        a2n = int(and_sort[1])
        a3n = int(and_sort[2])
        
        if len(ans[ans["着"]==a2n].index.values)>1:
            ans2nd = int(ans[ans["着"]==a2n].index.values[0])
            ans3rd = int(ans[ans["着"]==a2n].index.values[1])
        else:
            if i > 1:
                ans2nd = int(ans[ans["着"]==a2n].index.values[0])
            if i > 2:
                ans3rd = int(ans[ans["着"]==a3n].index.values[0])
    
    if ans1st==result1st:
        #print(ans1st,result1st)
        solo_count = solo_count+1

    if i > 1:
        if (ans1st==result1st)&(ans2nd==result2nd):
            doub_count = doub_count+1

    if i > 2:
        if (ans1st==result1st)&(ans2nd==result2nd)&(ans3rd==result3rd):
            tri_count = tri_count+1 
    
    if i > 2:
        ans_set = set([ans1st, ans2nd, ans3rd])
        res_set = set([result1st, result2nd,result3rd])
        if ans_set == res_set:
            trio_count = trio_count+1


    j = j + i
print("単勝的中率：",round(solo_count/len(val_group)*100,2),"%")
print("２連単的中率：",round(doub_count/len(val_group)*100,2),"%")
print("３連単的中率：",round(tri_count/len(val_group)*100,2),"%")
print("３連複的中率：",round(trio_count/len(val_group)*100,2),"%")


単勝的中率： 57.72 %
２連単的中率： 25.09 %
３連単的中率： 11.02 %
３連複的中率： 21.01 %
